[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/eight_conics/eight_conics.ipynb)

# Penrose's Eight-Conic Theorem

This example computes Penrose's eight-conic theorem, which completes a pattern of conics touching in pairs. Two conics in double contact touch at two points with common tangents there. Arranged on the corners of a cube, with neighbours in double contact and the four contact chords of every face meeting in a point, seven such conics admit an eighth that touches its three neighbours in double contact as well.

A conic is modelled as a map from points to their polar lines, `Conic = Plane <- Point`, which takes a point of the curve to its tangent. Adding the dyad of a line, `chord * (chord & Point)`, changes the conic everywhere except on that line, so the new conic meets the line at the same two points with the same tangents: double contact is an addition of maps. The eighth conic follows from each of its three neighbours by adding one more contact dyad, and the theorem is that the three sums are the same map.

In matrix notation a conic reads as a symmetric $3 \times 3$ matrix $C$, double contact along the line $\ell$ as the pencil $C + \lambda\,\ell\ell^{\mathsf T}$, and the theorem as three such pencils through the eighth conic's matrix.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga
from examples.animation import save_figure, show_animation

from examples.math.eight_conics import render

np.set_printoptions(precision=3, suppress=True)

In [ ]:
mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()
Point = ga.gatype.antivector()
Conic = ga.gatype((Plane, Point))
DualConic = ga.gatype((Point, Plane))

## 1. Double contact

A circle with the dyad of a line added, `circle + strength * chord * (chord & Point)`, is another conic, which agrees with the circle at every point of the line, since there the dyad pairs to nothing. The two meet the line at the same two points with the same polar lines, the tangents there, whatever the strength: they are in double contact. In the figure the circle is blue and the conic orange, through the same two points of their dashed chord.

In [ ]:
# The strength of the dyad, and the chord: off the circle's centre, and weighted.
strength = 2.5
chord_offset = 0.5
chord_weight = 0.7

center = mv.w.dual()                                                            # Point
semi_axes = stack((mv.x.dual(), mv.y.dual()))                             # [2] Point
chord = (mv.x - mv.w * chord_offset) * chord_weight                             # Plane

In [ ]:
def intersections(conics: Conic, lines: Plane) -> Point:
    """The two real crossings of each conic with its line, on the last batch axis."""
    origin = mv.w.dual()
    lines = lines.normalized()
    # Reflection gives the foot of the perpendicular from the chart origin.
    anchor = (origin + (lines >> origin)) / 2                                # [...] Point
    direction = (lines | mv.xy).dual()                                      # [...] Point
    quadratic = conics(direction) & direction                               # [...] Scalar
    linear = conics(anchor) & direction                                     # [...] Scalar
    constant = conics(anchor) & anchor                                      # [...] Scalar
    root_signs = np.array([-1, 1])
    distances = (-linear[..., None]
                 + (linear.squared() - quadratic * constant).square_root()[..., None]
                 * root_signs) / quadratic[..., None]                       # [..., 2] Scalar
    return anchor[..., None] + direction[..., None] * distances               # [..., 2] Point


# The dual conic maps tangent lines to their points of contact.
dual_circle = ((semi_axes * (Plane & semi_axes)).sum(axis=0)                    # DualConic
                         - center * (Plane & center))                    # [] Point <- Plane
circle = dual_circle.inverse()                                          # [] Plane <- Point

# This dyad pairs to nothing with every point of the chord: added, it keeps the circle's two crossings
# with the chord and their polar lines.
double_line = chord * (chord & Point)                                    # [] Plane <- Point
touching = circle + strength * double_line                              # [] Plane <- Point
contacts = intersections(circle, chord)                                 # [2] Point
print("the contacts on the other conic:", (touching(contacts) & contacts).to_array())
print("the tangents there, less the circle's:", (touching(contacts) - circle(contacts)).norm().to_array())
save_figure(render.draw_double_contact(circle, touching, chord, contacts), "eight_conics_double_contact");

## 2. Compatible contacts

The conics occupy the vertices of a cube whose edges record double contact. Each face must also have its four contact chords pass through one point. The cube describes the relationships between curves in the plane. Three chord choices and three coupling parameters build seven conics satisfying these conditions.

In [ ]:
# The edges' three chords, by their turn and their offset from the centre, and their weight: strong chords
# make the eight conics distinct ellipses, stronger ones turn some into hyperbolas. The three face
# couplings, swept around a loop a third of a turn apart, the frames around it and their duration.
chord_angles = np.arange(3) * (2 * np.pi / 3)
chord_offsets = np.array([0.20, 0.24, 0.16])
cube_chord_weight = 1.3
mean_couplings = np.array([0.30, -0.24, 0.27])
coupling_amplitude = 0.25
n_frames = 96
duration_ms = 90
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
couplings = mean_couplings + coupling_amplitude * np.cos(phases[:, None] + chord_angles)

# Each face pairs the other two chords.
forward = np.array([1, 2, 0])
backward = np.array([2, 0, 1])
# Binary vertex labels differ by one bit along each edge.
edges = np.array([[0, 1], [0, 2], [0, 4], [1, 3], [2, 3], [1, 5],
                  [4, 5], [2, 6], [4, 6], [3, 7], [5, 7], [6, 7]])
# The edges joining the eighth conic to its three neighbours.
closing_edges = np.array([9, 10, 11])

normals = (mv.xy * (-chord_angles / 2)).exp() >> mv.x                     # [3] Plane
chords = (normals - mv.w * chord_offsets) * cube_chord_weight             # [3] Plane

In [ ]:
double_lines = chords * (chords & Point)                                # [3] Plane <- Point
neighbours = circle + double_lines                                      # [3] Plane <- Point
left = chords[forward]                                                 # [3] Plane
right = chords[backward]                                               # [3] Plane
face_scales = 1 - couplings ** 2

# The cross terms couple two chord choices. Each resulting conic
# touches both neighbours, with all four contact chords concurrent.
cross_terms = left * (right & Point) + right * (left & Point)            # [3] Plane <- Point
faces = (circle * face_scales + double_lines[forward]                           # Conic
               + double_lines[backward] + cross_terms * couplings)     # [frames, 3] Plane <- Point
left_chords = left * couplings + right                                  # [frames, 3] Plane
right_chords = left + right * couplings                                  # [frames, 3] Plane

## 3. The eighth conic

The remaining three edges must reach the same conic. Each conic adjacent to the missing vertex supplies a candidate, obtained by adding a contact dyad with the appropriate scale. All three candidates agree throughout the deformation.

In [ ]:
closing_scale = 1 - np.sum(couplings ** 2, axis=-1) - 2 * np.prod(couplings, axis=-1)
closing_chords = (                                                              # Plane
    chords * face_scales
    + left * (couplings[:, backward] + couplings * couplings[:, forward])
    + right * (couplings[:, forward] + couplings * couplings[:, backward])
)                                                                      # [frames, 3] Plane

# Each candidate is in double contact with its corresponding face conic.
# Agreement of the three maps closes all three remaining edges at once.
completions = ((faces * closing_scale[:, None]                                  # Conic
                      + closing_chords * (closing_chords & Point))
                     / face_scales)                                    # [frames, 3] Plane <- Point
eighth = completions[:, 0]                                              # [frames] Plane <- Point

conics = stack((circle.broadcast_to((n_frames,)),                               # Conic
    neighbours[0].broadcast_to((n_frames,)), neighbours[1].broadcast_to((n_frames,)),
    faces[:, 2], neighbours[2].broadcast_to((n_frames,)), faces[:, 1], faces[:, 0],
    eighth), axis=-1)                                                  # [frames, 8] Plane <- Point
edge_chords = stack((chords[0].broadcast_to((n_frames,)),                       # Plane
    chords[1].broadcast_to((n_frames,)), chords[2].broadcast_to((n_frames,)),
    left_chords[:, 2], right_chords[:, 2], right_chords[:, 1], left_chords[:, 1],
    left_chords[:, 0], right_chords[:, 0], closing_chords[:, 2],
    closing_chords[:, 1], closing_chords[:, 0]), axis=-1)                # [frames, 12] Plane
contacts = intersections(conics[:, edges[:, 0]], edge_chords)           # [frames, 12, 2] Point
show_animation(render.animate_eighth(conics, edge_chords, contacts, edges, closing_edges), "eight_conics_eighth",
               duration_ms)

The eighth conic is drawn bold in orange and its three neighbours in blue, green and pink, each touching it at two points of their dashed common chord; the other four conics are faint. As the couplings change, all eight conics move, and the eighth keeps all six of its contacts.

In [ ]:
# Agreement as maps checks every input point, not only the drawn contacts.
print("Eighth conic")
print(eighth[0].coefficient_table())
print("Differences from the other two constructions")
for difference in completions[0, 1:] - eighth[0]:
    print(difference.coefficient_table())

In [ ]:
# checks
# The three constructions of the eighth conic are one map, at every coupling of the deformation.
np.testing.assert_allclose((completions[:, 1:] - completions[:, :1]).kernel, 0, atol=1e-12)

The closure is an identity between sums of contact dyads: the same map carries all three tangency requirements. The construction follows the Penrose formulas in §6 of Arnold, Chern, Eide, Gunn, Neukirchner and Penrose's [*Penrose's eight-conic theorem*](https://arxiv.org/html/2409.17150v8#S6). Their theorem also covers configurations beyond the ellipses shown here.